# Resting EEG Feature Extraction

This notebook turns the cleaned resting-state EEG epochs from `03_Batch_Preprocessing.ipynb` into one feature table for downstream modelling or statistical analysis.

The implementation details live in `src/eeg_feature_extraction.py`. This notebook is organised as a walkthrough: first we check the input data, then inspect what each feature family means, then run extraction and save the final table.

## 1. Setup

This cell sets paths, imports the feature-extraction helpers, and keeps the notebook connected to the local `src/` package. If you edit the script, rerun this cell so the notebook picks up the new version.

In [45]:
from importlib import reload
from pathlib import Path
import os
import sys

import mne
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))
mne.set_log_level("WARNING")

import src.eeg_feature_extraction as efe

reload(efe)

processed_eeg_dir = project_root / "data" / "processed" / "eeg"
features_dir = project_root / "data" / "features"

print(f"Project root: {project_root}")
print(f"Processed EEG dir: {processed_eeg_dir}")
print(f"Features dir: {features_dir}")

Project root: /Users/nataliemarryatt/neurogenetics-ml
Processed EEG dir: /Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg
Features dir: /Users/nataliemarryatt/neurogenetics-ml/data/features


## 2. Check Dependencies

The required packages handle EEG loading and tabular output. Optional packages enable the more specialised feature families: aperiodic modelling, connectivity, graph metrics, complexity, and microstates.

If a package is missing, the corresponding optional feature family is skipped by the helper functions rather than crashing the whole notebook.

In [46]:
efe.dependency_table()

,package,available,purpose
0,mne,True,core EEG I/O and spectral estimation
1,numpy,True,numerical arrays
2,pandas,True,feature tables
3,scipy,True,signal summaries and integration
4,mne_connectivity,True,connectivity matrices
5,specparam,True,aperiodic / 1/f spectral parameterisation
6,antropy,True,entropy and fractal-complexity features
7,pycrostates,True,microstate clustering and backfitting


## 3. Locate Clean Resting EEG Epochs

The preprocessing notebook saves one cleaned epoch file per subject and condition:

- `eyes_open`: resting EEG recorded with eyes open
- `eyes_closed`: resting EEG recorded with eyes closed

Each row below is one feature-extraction input. Later, each row will become one row in the feature table.

In [47]:
epoch_index = efe.find_clean_epoch_files(processed_eeg_dir)

print(f"Found {len(epoch_index)} clean epoch files across {epoch_index['subject_id'].nunique() if not epoch_index.empty else 0} subjects.")
epoch_index

Found 8 clean epoch files across 4 subjects.


,subject_id,condition,path
0,sub-01,eyes_closed,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-01/eeg/sub-01_task-rest_eyes_closed_clean-epo.fif
1,sub-01,eyes_open,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-01/eeg/sub-01_task-rest_eyes_open_clean-epo.fif
2,sub-02,eyes_closed,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-02/eeg/sub-02_task-rest_eyes_closed_clean-epo.fif
3,sub-02,eyes_open,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-02/eeg/sub-02_task-rest_eyes_open_clean-epo.fif
4,sub-03,eyes_closed,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-03/eeg/sub-03_task-rest_eyes_closed_clean-epo.fif
5,sub-03,eyes_open,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-03/eeg/sub-03_task-rest_eyes_open_clean-epo.fif
6,sub-04,eyes_closed,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-04/eeg/sub-04_task-rest_eyes_closed_clean-epo.fif
7,sub-04,eyes_open,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-04/eeg/sub-04_task-rest_eyes_open_clean-epo.fif


## 4. Inspect What We Are Extracting From

Before calculating features, check that each file has a reasonable number of clean epochs and EEG channels.

What to look at:
- `n_epochs`: how many clean fixed-length segments survived preprocessing and rejection
- `n_channels`: how many usable EEG channels remain after excluding bad channels
- `duration_minutes`: approximate amount of clean EEG contributing to the features

Large differences here matter because feature reliability depends on how much clean data each participant contributes.

In [48]:
epoch_summary = efe.summarize_epoch_files(epoch_index)
epoch_summary

,subject_id,condition,n_epochs,n_channels,sfreq,duration_minutes
0,sub-01,eyes_closed,177,122,250.0,5.888200
1,sub-01,eyes_open,119,124,250.0,3.958733
2,sub-02,eyes_closed,179,126,250.0,5.954733
3,sub-02,eyes_open,118,125,250.0,3.925467
4,sub-03,eyes_closed,180,126,250.0,5.988000
5,sub-03,eyes_open,120,127,250.0,3.992000
6,sub-04,eyes_closed,178,126,250.0,5.921467
7,sub-04,eyes_open,120,125,250.0,3.992000


## 5. Feature Families and What They Mean

This table is the conceptual map for the extraction step. The final dataset contains many columns, but they come from a small number of feature families.

For the article, these are also useful categories for describing the EEG feature extraction approach.

In [49]:
pd.set_option("display.max_colwidth", 120)
efe.feature_family_table()

,family,what_it_summarises,main_metrics,interpretation
0,Spectral power,How EEG power is distributed across frequency bands.,"absolute power, relative power, regional power, peak alpha frequency, theta/beta ratio",Higher values mean stronger activity in that band or region; relative power controls for total 1-40 Hz power.
1,Aperiodic / 1/f,"The broadband background shape of the power spectrum, separated from narrow oscillatory peaks.","aperiodic offset, aperiodic exponent, oscillatory peak frequency/power/bandwidth",The exponent reflects spectral slope; peak metrics describe rhythmic components after modelling the background.
2,Time-domain,Basic properties of the cleaned EEG waveform.,"standard deviation, peak-to-peak amplitude, line length, skew, kurtosis, Hjorth parameters",Useful signal summaries and QC-adjacent features; large amplitudes can reflect neural signal or residual artifact.
3,Connectivity,Synchronisation between EEG channels within each frequency band.,"mean wPLI, standard deviation of wPLI","wPLI reduces zero-lag volume-conduction effects; with 2-second epochs this notebook estimates theta and above, not d..."
4,Graph theory,Network organisation after thresholding the connectivity matrix.,"density, mean strength, mean clustering",These compress pairwise connectivity into network-level summaries.
5,Complexity,Irregularity and fractal structure of the resting EEG signal.,"permutation entropy, spectral entropy, Higuchi fractal dimension",Higher entropy generally means less predictable signal structure.
6,Microstates,"Short-lived, recurring scalp topographies and their transitions.","duration, occurrence, coverage, transition probabilities, global explained variance","Fit templates on training subjects only, then backfit held-out subjects to avoid leakage."


## 6. Inspect One Example Participant

Before running the whole cohort, extract features from one file. This makes the output easier to understand.

The feature values below are not interpreted as group effects. They are a sanity check showing the structure of one subject-condition row.

In [50]:
example_row = epoch_index.iloc[0]
example_epochs = efe.load_epochs(example_row["path"])

print(f"Example: {example_row['subject_id']} | {example_row['condition']}")
print(f"Epochs: {len(example_epochs)}")
print(f"Sampling frequency: {example_epochs.info['sfreq']} Hz")

Example: sub-01 | eyes_closed
Epochs: 177
Sampling frequency: 250.0 Hz


### 6.1 Spectral Features

Spectral features summarise power in frequency bands. For resting EEG, these are usually the most central features.

For this one example participant, the code below shows the main calculation: estimate the power spectral density, integrate power within a band, then summarise across epochs and channels. The reusable function in `src/eeg_feature_extraction.py` repeats this same logic for all bands and regions.

Metrics to notice:
- `power_abs_*`: absolute bandpower in the EEG signal
- `power_rel_*`: that band as a proportion of total 1-40 Hz power
- `peak_alpha_frequency_posterior`: strongest posterior alpha frequency, often relevant in resting eyes-closed EEG
- `theta_beta_ratio_global`: global theta power divided by global beta power

In [51]:
from scipy.integrate import simpson

n_per_seg = int(round(example_epochs.info["sfreq"] * efe.WELCH_WINDOW_S))

spectrum = example_epochs.compute_psd(
    method="welch",
    fmin=efe.PSD_FMIN,
    fmax=efe.PSD_FMAX,
    picks="eeg",
    exclude="bads",
    n_per_seg=n_per_seg,
    n_fft=n_per_seg,
)

psd = spectrum.get_data()  # shape: epochs x channels x frequencies
freqs = spectrum.freqs

alpha_mask = (freqs >= 8) & (freqs < 13)
alpha_power = simpson(psd[..., alpha_mask], x=freqs[alpha_mask], axis=-1)

total_mask = (freqs >= 1) & (freqs < 40)
total_power = simpson(psd[..., total_mask], x=freqs[total_mask], axis=-1)

example_spectral_summary = pd.Series(
    {
        "alpha_absolute_power_global": alpha_power.mean(),
        "alpha_relative_power_global": (alpha_power / total_power).mean(),
        "n_epochs": psd.shape[0],
        "n_channels": psd.shape[1],
        "n_frequency_bins": psd.shape[2],
    }
)

example_spectral_summary.to_frame("value")

,value
alpha_absolute_power_global,1.732969e-11
alpha_relative_power_global,3.894909e-01
n_epochs,1.770000e+02
n_channels,1.220000e+02
n_frequency_bins,7.900000e+01


### 6.2 Aperiodic Features

Aperiodic features separate the broadband 1/f-like background from narrow oscillatory peaks. This is important because an apparent bandpower difference may reflect a broad spectral slope change rather than a true change in a rhythmic oscillation.

For one participant, the important steps are: average the PSD across epochs and channels, fit a spectral model, then pull out the fitted background slope and peak parameters.

Metrics to notice:
- `aperiodic_offset`: overall vertical position of the fitted background spectrum
- `aperiodic_exponent`: steepness of the spectral slope
- `n_oscillatory_peaks`: number of fitted rhythmic peaks
- `strongest_peak_*`: frequency, power, and bandwidth of the strongest fitted peak

In [52]:
from specparam import SpectralModel

mean_psd = psd.mean(axis=(0, 1))

spectral_model = SpectralModel(
    peak_width_limits=(1.0, 8.0),
    max_n_peaks=6,
    verbose=False,
)
spectral_model.fit(freqs, mean_psd, [efe.PSD_FMIN, efe.PSD_FMAX])

param_dict = spectral_model.results.params.asdict()
aperiodic_params = param_dict["aperiodic_fit"]
peak_params = param_dict["peak_fit"]

example_aperiodic_summary = pd.Series(
    {
        "aperiodic_offset": aperiodic_params[0],
        "aperiodic_exponent": aperiodic_params[1],
        "n_oscillatory_peaks": len(peak_params),
    }
)

example_aperiodic_summary.to_frame("value")

,value
aperiodic_offset,-11.270395
aperiodic_exponent,0.990047
n_oscillatory_peaks,3.000000


### 6.3 Time-Domain and Complexity Features

These features describe waveform shape and signal irregularity. They are useful as compact signal summaries, but should be interpreted cautiously because residual artifacts can also influence them.

For one participant, the main operation is to work directly with the cleaned EEG array: epochs x channels x time points. Hjorth features use the signal variance and the variance of its first and second differences.

Metrics to notice:
- `td_sd_uv`, `td_peak_to_peak_uv`: signal amplitude summaries
- `td_line_length_uv`: total signal movement across time; high values can reflect fast activity or artifact
- `hjorth_activity`: variance of the signal
- `hjorth_mobility`: how quickly the signal changes
- `hjorth_complexity`: how much the signal shape deviates from a simple sine-like waveform
- entropy/fractal metrics: irregularity and complexity of the signal

In [53]:
import antropy as ant

data_uv = example_epochs.copy().pick("eeg", exclude="bads").get_data() * 1e6
first_diff = np.diff(data_uv, axis=-1)
second_diff = np.diff(first_diff, axis=-1)

activity = np.var(data_uv, axis=-1)
mobility = np.sqrt(np.var(first_diff, axis=-1) / activity)
complexity = np.sqrt(np.var(second_diff, axis=-1) / np.var(first_diff, axis=-1)) / mobility

mean_channel_signal = data_uv.mean(axis=0)[0]

example_time_complexity_summary = pd.Series(
    {
        "signal_sd_uv": data_uv.std(),
        "mean_peak_to_peak_uv": np.ptp(data_uv, axis=-1).mean(),
        "hjorth_activity": np.nanmean(activity),
        "hjorth_mobility": np.nanmean(mobility),
        "hjorth_complexity": np.nanmean(complexity),
        "permutation_entropy_first_channel": ant.perm_entropy(mean_channel_signal, normalize=True),
    }
)

example_time_complexity_summary.to_frame("value")

,value
signal_sd_uv,6.766251
mean_peak_to_peak_uv,36.069826
hjorth_activity,45.545013
hjorth_mobility,0.320231
hjorth_complexity,1.983392
permutation_entropy_first_channel,0.770927


### 6.4 Connectivity and Graph Features

Connectivity features summarise synchronisation between channels. Here the default is wPLI, which focuses on phase-lagged coupling and is less sensitive to zero-lag volume conduction than ordinary coherence.

Because the current preprocessing uses 2-second epochs, this notebook estimates connectivity from theta upward. Delta connectivity would need longer epochs to contain enough low-frequency cycles.

Graph metrics are calculated after thresholding the connectivity matrix. These summarise network organisation rather than individual channel-pair connections.

For one participant, the main steps are: compute a channel-by-channel connectivity matrix for one frequency band, summarise the upper triangle, then threshold the matrix to make simple graph summaries.

Metrics to notice:
- `connectivity_wpli_*_mean`: average phase-lagged coupling in a frequency band
- `connectivity_wpli_*_sd`: variability of connectivity across channel pairs
- `graph_*_density`: proportion of retained network edges after thresholding
- `graph_*_strength_mean`: average weighted connection strength
- `graph_*_clustering_mean`: tendency for neighbouring nodes to form connected clusters

In [54]:
from mne_connectivity import spectral_connectivity_epochs

eeg_epochs = example_epochs.copy().pick("eeg", exclude="bads")

alpha_connectivity = spectral_connectivity_epochs(
    eeg_epochs,
    method="wpli",
    mode="multitaper",
    sfreq=example_epochs.info["sfreq"],
    fmin=8,
    fmax=13,
    faverage=True,
    verbose=False,
)

matrix = alpha_connectivity.get_data(output="dense")[:, :, 0]
matrix = np.maximum(matrix, matrix.T)
np.fill_diagonal(matrix, 0.0)

upper_triangle = matrix[np.triu_indices_from(matrix, k=1)]
threshold = np.nanpercentile(upper_triangle, 75)
adjacency = (matrix >= threshold).astype(float)
np.fill_diagonal(adjacency, 0.0)

example_connectivity_summary = pd.Series(
    {
        "alpha_wpli_mean": np.nanmean(upper_triangle),
        "alpha_wpli_sd": np.nanstd(upper_triangle),
        "alpha_graph_density": adjacency.sum() / (adjacency.shape[0] * (adjacency.shape[0] - 1)),
        "n_channels_in_matrix": matrix.shape[0],
    }
)

example_connectivity_summary.to_frame("value")

,value
alpha_wpli_mean,0.370249
alpha_wpli_sd,0.145416
alpha_graph_density,0.250102
n_channels_in_matrix,122.000000


### 6.5 Microstate Features

Microstates are relevant for resting EEG, but they need special handling in a machine-learning workflow. The templates should be learned only from training subjects, then applied to validation/test subjects. That avoids leaking information from held-out participants into the feature extraction step.

For that reason, this notebook sets up the dependency and documents the next step, but does not yet add microstate features to the saved table.

In [55]:
efe.microstate_note()

'Microstate extraction should be added after defining train/test or CV folds. Fit templates on training subjects only, then backfit held-out subjects to avoid leakage.'

## 7. Run Feature Extraction Across Available Files

Now each cleaned subject-condition file is converted into one row of features. With the current data layout, a participant can contribute one eyes-open row and one eyes-closed row.

The final table is still participant-level in the sense that features are averaged over epochs within each subject and condition. It is not treating each short epoch as an independent participant.

In [ ]:
features = efe.extract_all_features(epoch_index)

print(f"Feature table shape: {features.shape[0]} rows x {features.shape[1]} columns")
features.head()

Extracting sub-01 eyes_closed
Extracting sub-01 eyes_open
Extracting sub-02 eyes_closed


## 8. Quick Output Checks

Before saving, check the number of rows per condition and the amount of missing data. Missing columns usually mean an optional package was unavailable or a model did not detect a peak for one participant.

In [ ]:
display(features.groupby("condition")["subject_id"].nunique().to_frame("n_subjects"))

missing_summary = (
    features.isna()
    .mean()
    .sort_values(ascending=False)
    .loc[lambda s: s > 0]
    .to_frame("proportion_missing")
)
missing_summary.head(20)

## 9. Save Feature Table

The saved file is the main output of this notebook. It can be merged with genotype and demographic metadata in later modelling notebooks.

In [ ]:
feature_path = efe.save_features(features, features_dir)
print(f"Saved {features.shape[0]} rows and {features.shape[1]} columns to {feature_path}")